## RAG

### [Milvus](https://milvus.io/api-reference/pymilvus/v3.0.x/About.md)

#### 基本用法

In [ ]:
from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

##### 1、Database

In [ ]:
print(client.list_databases())

In [ ]:
db_name = "rag_test"

client.create_database(db_name)

In [ ]:
client.describe_database(db_name)

In [ ]:
client.drop_database(db_name)

##### 2、Collection

In [ ]:
client.use_database(db_name)

In [ ]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE",
)

In [ ]:
client.list_collections()

In [ ]:
client.describe_collection(collection_name)

In [ ]:
client.drop_collection(collection_name)

#### DML 操作

##### 1、嵌入模型初始化

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

In [ ]:
import os
from langchain.embeddings import init_embeddings

embeddings = init_embeddings(
    model="openai:text-embedding-v4",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_API_BASE"),
    check_embedding_ctx_length=False,
)

##### 2、写数据

In [ ]:
texts = [
    "Redis 是内存键值数据库，经常被用于缓存。",
    "缓存热点数据可以减少对数据库的访问。",
    "合理的过期时间有助于控制缓存数据的新鲜度。",
    "MySQL 事务可以控制一组操作的提交和回滚。",
    "InnoDB 使用 undo 日志支持事务回滚和一致性读取。",
    "事务隔离级别会影响并发读写时能够看到的数据。",
]

In [ ]:
vectors = embeddings.embed_documents(texts)

print(len(vectors))
print(len(vectors[0]))

In [ ]:
# 封装为写入数据
data = [
    {
        "id": i,
        "vector": vectors[i],
        "text": texts[i],
        "source": "test"
    } for i in range(len(vectors))
]

client.upsert(collection_name, data)

In [ ]:
# 手动
client.flush(collection_name)

In [ ]:
client.get_collection_stats(collection_name)

#### DQL 操作

##### 1、主键查询

In [ ]:
rows = client.get(
    collection_name=collection_name,
    ids=[0, 1, 2]
)

for row in rows:
    print(row["id"], ": ", row["text"], sep="")

##### 2、条件查询

In [ ]:
rows = client.query(
    collection_name=collection_name,
    filter="id > 3",
    limit=2,
    consistency_level="Strong",
)

for row in rows:
    print(row["id"], ": ", row["text"], sep="")

In [ ]:
iterator = client.query_iterator(
    collection_name=collection_name,
    filter="id > 3",
    limit=2,
    output_fields=["*"],
)

while True:
    rows = iterator.next()

    # 为空则遍历结束
    if not rows:
        iterator.close()
        break

    for row in rows:
        print(row["id"], ": ", row["text"], sep="")

##### 3、向量查询

In [ ]:
query_vector = embeddings.embed_query("redis 是什么")
rows = client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=3,
    output_fields=["id", "text"],
)

for row in rows[0]:
    print(row)